# Load modules

In [1]:
import os
import itk
import cv2
import sys
import json
import glob
import time
import imageio
import numpy as np
import pandas as pd
import nibabel as nib
import pydicom as dcm
from matplotlib import colors
import matplotlib.pyplot as plt

from joblib import Parallel, delayed
from lmfit import minimize, Parameters

from urllib.request import urlopen
from datetime import datetime as dtime



# Auxiliar Functions
## Utils

In [40]:
def getenv():
    """
    Requires sys and os modules:
    import sys
    import os
    """
    if sys.platform == 'win32':
        env_home = 'HOMEPATH'
    elif (sys.platform == 'darwin') | (sys.platform == 'linux'):
        env_home = 'HOME'
    HOMEPATH = os.getenv(env_home)
    
    return HOMEPATH

def check_path_exist(path, file=False):
    """
    Flag FILE indicates the path contains a file name (FLAG=TRUE) or the path only points to a folder (FLAG=FALSE (Default))
    """
    if file:
        is_path = os.path.isfile(path)
    else:
        is_path = os.path.isdir(path)

    print(f'{"OK:" if is_path else "ERROR:"} Path to {"file" if file else "folder"} {path} does{"" if is_path else " NOT"} exist')

    return is_path

def list_folder(path, skip_patterns=None, sorted=False):
    # Check the path exists:
    if not(os.path.isdir(path)):
        error_message = f'[ERROR]: Folder path {path} does not exist'
        sys.exit(error_message)
    
    raw_list = os.listdir(path)
    
    # Ensure patterns is a list, defaulting to ['.DS_Store']
    dsStorePattern = '.DS_Store'
    if skip_patterns is None:
        skip_patterns = [dsStorePattern]
    elif isinstance(skip_patterns, str):
        skip_patterns = [dsStorePattern, skip_patterns]  # Convert single string to list
    elif isinstance(skip_patterns, (list, tuple, set)):  
        skip_patterns = list(skip_patterns)  # Ensure it's a list
        if dsStorePattern not in skip_patterns:
            skip_patterns.append(dsStorePattern)  # Add default pattern if not already included
    else:
        raise TypeError("patterns must be a string, list, tuple, or set.")

    # dir_list = [item for item in raw_list if skip_pattern not in item]
    # any(p in item for p in skip_patterns) checks if any pattern in skip_patterns exists in item:
    dir_list = [item for item in raw_list if not any(p in item for p in skip_patterns)]

    if sorted:
        dir_list.sort()

    return dir_list


# Function to parse mixed datetime formats
def parse_mixed_datetime(ts, format='%Y%m%d%H%M%S.%f'):
    """
    # Ensure all values have milliseconds (force `.0000` if missing) before the conversion:
    """

    ts_f = ts.apply(lambda x: f"{x}.0000" if pd.notna(x) and "." not in str(x) else x)

    return pd.to_datetime(ts_f, format=format, errors="coerce")

def filter_list(lst, pattern):
    return [item for item in lst if pattern not in item]

def get_list_element(lst, pattern):
    return [item for item in lst if pattern in item]


## Enrst Angles and SPGR equations

In [41]:
def E1(TR, T1):
    
    return np.exp(-TR/T1)

def E2(TE, T2):
    
    return np.exp(-TE/T2)

def fzss(TR, T1, FA_rad):
    
    return (1.0 - E1(TR, T1))/(1.0 - np.cos(FA_rad)*E1(TR, T1))

def s0sinalfa(M0, FA_rad):

    return M0 * np.sin(FA_rad)

def st(M0, T1, TR, FA_rad, TE=np.nan, T2=np.nan):

    stT1 = s0sinalfa(M0, FA_rad) * fzss(TR, T1, FA_rad)
    
    if (np.isnan(T2)) | (np.isnan(TE)):
        return stT1
    else:
        #eq  with t2* decay
        return stT1 * E2(TE,T2)

def ernstAngle(TR, T1):
    # Flip Angle that maximises the signal for a given TR and T1 (REF: Handbook of MRI Pulse Sequences, page 587, Eq. 14.9)
    
    return np.rad2deg(np.arccos(E1(TR, T1)))

def anglesT1Maps(TR,T1):
    # Flip angles for optimal T1 Mapping with only 2 acq (REF: Deoni SCL 2003, Rapid Combined T1 and T2 mapping using..., MRM 49:515-526)
    
    f = 0.71 # see REF
    angle1 = (E1(TR, T1)*np.square(f) + (1-np.square(E1(TR, T1)))*np.sqrt(1-np.square(f)))/(1-(np.square(E1(TR, T1))*(1-np.square(f))))
    angle2 = (E1(TR, T1)*np.square(f) - (1-np.square(E1(TR, T1)))*np.sqrt(1-np.square(f)))/(1-(np.square(E1(TR, T1))*(1-np.square(f))))
    Angles = np.array([angle1,angle2])
    #return Angles
    return np.rad2deg(np.arccos(Angles))


## Helper functions for Linear T1 mapping

In [42]:
# Define a simple atomic function to perform linear fit 
# Add a check before fitting
def safe_polyfit(x, y):
    if np.any(np.isnan(x)) or np.any(np.isnan(y)):
        return np.nan, np.nan
    if np.all(x == x[0]) or np.all(y == y[0]):
        return np.nan, np.nan
    if np.any(x == 0.0) or np.any(y == 0.0):
        return np.nan, np.nan    
    try:
        m, n = np.polyfit(x, y, deg=1)
        return m, n
    except Exception:
        return np.nan, np.nan
    
# Function to apply polyfit to a single pixel across the flip angles
def linfit_pixel(X, Y, i, j, k):
    x_vals = X[:, k, j, i]
    y_vals = Y[:, k, j, i]

    # try:
    mslope, nintercept = safe_polyfit(x_vals, y_vals)
    # except Exception:
    #     mslope = np.nan
    #     nintercept = np.nan

    return (i, j, k, mslope, nintercept)


## Optimisation functions for nonlinear T1 map estimation

In [43]:
# Define cost functions to estimate T1 with SPGR and IR
def fit_SPGR_t1(params, flip_angle_rad, values):
    s0 = params['s0'].value
    t1 = params['t1'].value
    tr = params['tr'].value
    te = params['te'].value
    t2star = params['t2star'].value

    s = st(s0, t1, tr, flip_angle_rad, te, t2star)
    
    return (np.fabs(s) - np.fabs(values))

# Define the optimisation method
def do_spgr_fit(ydata, fa_rad, tr, te=np.nan, t2star=np.nan, initial_s0=1.0, initial_t1=1000.0, T1ub = 10000.0):
    params = Parameters()
    params.add('s0', value=initial_s0, vary=True, min=0.0)
    params.add('t1', value=initial_t1, vary=True, min=0.0, max=T1ub)
    params.add('tr', value=tr, vary=False)
    params.add('te', value=te, vary=False)
    params.add('t2star', value=t2star, vary=False)
    
    result = minimize(fit_SPGR_t1, params, args=(fa_rad, ydata))

    return result    

def safe_nonlinfit(ydata, flipangles_rad, m0, t10, TR, TE, T2star, T1ub):

    if np.any(np.isnan(ydata)):
        return np.nan, np.nan
    
    if np.all(ydata == ydata[0]):
        return np.nan, np.nan
        
    try:
        nonlin_fit_output = do_spgr_fit(ydata, flipangles_rad, TR, TE, T2star, 
                                        T1ub=T1ub,
                                        initial_s0=m0, initial_t1=t10)
        return nonlin_fit_output.params['t1'].value, nonlin_fit_output.params['s0'].value    

    except Exception:
        return np.nan, np.nan
 
# Function to apply nonlinear fit to a single pixel across the flip angles
def nonlinfit_pixel(Y, FlipAngleRad, T10, TR, T1ub, i, j, k, TE=np.nan, T2star=np.nan):
    ydata = Y[:, k, j, i]

    t1hat, m0hat = safe_nonlinfit(ydata, FlipAngleRad, m0=np.max(ydata), t10=T10, TR=TR, TE=TE, T2star=T2star, T1ub=T1ub)

    return (i, j, k, t1hat, m0hat)


## T1 mapping algorithms

In [44]:
def t1fit_vfa(VFA_listOfImages, VFA_listOfFlipAngles, VFA_parameters, T10=1000.0, FitAlgo='rational_linear', flip_angle_units = 'deg', T1ub=5000.0, T1lb=100.0):
    # Algorithms can be:
    # rational_linear --> Rational Approximation (Helms et al., 2008)
    # std_linear    --> Linear approximation (Gutpa, 1977)
    # non_linear    --> Non-linear ("exact" fitting)

    # List of images must match the length of the flip angles:
    nImages = len(VFA_listOfImages)
    nFlipAngles = len(VFA_listOfFlipAngles)

    if nImages != nFlipAngles:
        print(f'[ERROR]: Length of images {nImages} does not match the list of flip angles {nFlipAngles}')
        sys.exit()

    if nImages < 2:
        print(f'[ERROR]: Not enough images {len(nImages)} to perform the calculations')
        sys.exit()

    # At least we need the TR for the fitting
    if 'TR' not in VFA_parameters.keys():
        print('[ERROR]: Parameter TR is not available, cannot perform the fitting')
        sys.exit()
    
    # If required, convert the flip angles from degree to radians (default behaviour)
    if flip_angle_units == 'deg':
        flip_angles_radians = [np.deg2rad(x) for x in VFA_listOfFlipAngles]
    else:
        flip_angles_radians = VFA_listOfFlipAngles.copy()

    # Call the fitting function 
    if FitAlgo == 'rational_linear':
        if nImages > 2:
            print('[WARNING]: Algorithm {FitAlgo} requires only 2 Flip angles, but {nImages} were provided')
            print(f'Will use only the first 2 flip angles ({flip_angles_radians[:2]})')
            nImages = 2
        T1, M0 = rational_linear_t1_fit(VFA_listOfImages[:nImages], flip_angles_radians[:nImages], VFA_parameters['TR'], T1ub=T1ub, T1lb=T1lb)
    elif FitAlgo == 'std_linear':
        T1 = 0
        M0 = 0
        T1, M0 = standard_linear_t1_fit(VFA_listOfImages[:nImages], flip_angles_radians[:nImages], VFA_parameters['TR'], T1ub=T1ub, T1lb=T1lb)
    elif FitAlgo == 'non_linear' :
        T1 = 0
        M0 = 0
        T1, M0 = nonlinear_t1_fit(VFA_listOfImages[:nImages], flip_angles_radians[:nImages], T10, VFA_parameters['TR'], T1ub=T1ub)
    else:
        print(f'[ERROR]: Method {FitAlgo} is not yet implemented')
        T1 = []
        M0 = []
        sys.exit()
    
    return T1, M0

def rational_linear_t1_fit(VFA_images, VFA_flipangles_rad, TR, T1ub, T1lb):
    
    if 0.0 in VFA_flipangles_rad:
        print('[ERROR]: One of the angles is 0, cannot perform the calculations')
        sys.exit()

    Tnum = (VFA_images[0] / VFA_flipangles_rad[0]) - (VFA_images[1] / VFA_flipangles_rad[1]) 
    Tden = (VFA_images[1] * VFA_flipangles_rad[1]) - (VFA_images[0] * VFA_flipangles_rad[0])
    Tden += 1e-6

    T1map = 2.0 * TR * Tnum / Tden

    # T1map[np.isnan(T1map)] = 1.0
    T1map[T1map > T1ub] = 0.0
    T1map[T1map < T1lb] = 0.0

    Anum = (VFA_flipangles_rad[1]/VFA_flipangles_rad[0] - VFA_flipangles_rad[0]/VFA_flipangles_rad[1])
    Aapp = VFA_images[0] * VFA_images[1] * Anum / Tden
    # Aapp[np.isnan(Aapp)] = 0.0
    Aapp[Aapp < 0.0] = 0.0

    return T1map, Aapp

def standard_linear_t1_fit(VFA_images, VFA_flipangles_rad, TR, T1ub, T1lb):

    # Create the xi and yi elements for the X and Y lists (length is the number of flip angles) and 
    # convert it to an np.array for easier manipulation (shape: flipAngle, slices, height, width)
    X = np.array([fa_image/np.tan(fa_value) for fa_image, fa_value in zip(VFA_images, VFA_flipangles_rad)])
    Y = np.array([fa_image/np.sin(fa_value) for fa_image, fa_value in zip(VFA_images, VFA_flipangles_rad)])
    if len(X.shape) == 1:
        X = X[:, np.newaxis, np.newaxis, np.newaxis]
        Y = Y[:, np.newaxis, np.newaxis, np.newaxis]
    elif len(X.shape) == 2:
        X = X[:, :, np.newaxis, np.newaxis]
        Y = Y[:, :, np.newaxis, np.newaxis]
    elif len(X.shape) == 3:
        X = X[:, :, :, np.newaxis]
        Y = Y[:, :, :, np.newaxis]
    
    nFA, nS, nH, nW = X.shape

    # Flatten all pixel coordinates (order seems counter-intuitive, but it is following ITK convention)
    pixel_indices = [(k, j, i) for k in range(nS) for j in range(nH) for i in range(nW)]

    # Run in parallel
    results = Parallel(n_jobs=-1)(delayed(linfit_pixel)(X, Y, i, j, k) for k, j, i in pixel_indices)

    # Reconstruct the output maps (slope and intercepts volumes)
    m = np.zeros((nS, nH, nW))
    n = np.zeros((nS, nH, nW))
    for i, j, k, mijk, nijk in results:
        m[k, j, i] = mijk
        n[k, j, i] = nijk
    
    T1hat = -TR / np.log(m)
    T1hat[np.isnan(T1hat)] = 0.0
    T1hat[T1hat > T1ub] = 0.0
    T1hat[T1hat < T1lb] = 0.0
    
    M0hat = n/(1-m)

    return T1hat, M0hat

def nonlinear_t1_fit(VFA_images, VFA_flipangles_rad, T10, TR, T1ub):
    
    # def nonlinfit_pixel(Y, FlipAngle, i, j, k, TR, TE=None, T2start=None):
    
    # Create the yi elements for the Y list (length is the number of flip angles) and 
    # convert it to an np.array for easier manipulation (shape: flipAngle, slices, height, width)
    Y = np.array(VFA_images)
    if len(Y.shape) == 1:
        Y = Y[:, np.newaxis, np.newaxis, np.newaxis]
    elif len(Y.shape) == 2:
        Y = Y[:, :, np.newaxis, np.newaxis]
    elif len(Y.shape) == 3:
        Y = Y[:, :, :, np.newaxis]

    nFA, nS, nH, nW = Y.shape

    # Flatten all pixel coordinates (order seems counter-intuitive, but it is following ITK convention)
    pixel_indices = [(k, j, i) for k in range(nS) for j in range(nH) for i in range(nW)]

    # Run in parallel
    results = Parallel(n_jobs=-1)(delayed(nonlinfit_pixel)(Y, VFA_flipangles_rad, T10, TR, T1ub, i, j, k) for k, j, i in pixel_indices)

    # Reconstruct the output maps (slope and intercepts volumes)
    T1hat = np.zeros((nS, nH, nW))
    M0hat = np.zeros((nS, nH, nW))

    for i, j, k, t1est, m0est in results:
        T1hat[k, j, i] = t1est
        M0hat[k, j, i] = m0est

    T1hat[np.isnan(T1hat)] = 0.0
    # T1hat[T1hat > T1ub] = 0.0
    # T1hat[T1hat < 0.0] = 0.0

    return T1hat, M0hat


## Pre-processing T1 map

In [45]:
def nt4k_bias_correction(input_itk_volume, simple=False):

    # Cast to float if needed
    input_volume = itk.cast_image_filter(input_itk_volume, ttype=[type(input_itk_volume), itk.Image[itk.F, 3]])

    # Create a mask (you can use the whole image or a thresholded region)
    mask_image = itk.Image[itk.UC, 3].New()
    mask_image.SetRegions(input_volume.GetLargestPossibleRegion())
    mask_image.CopyInformation(input_volume)
    mask_image.Allocate()
    mask_image.FillBuffer(1)  # Whole image as mask

    # Initialize the N4 bias field correction filter
    n4_filter = itk.N4BiasFieldCorrectionImageFilter.New(Input=input_volume, MaskImage=mask_image)

    if not simple:
        # # Set the shrink factor (4 is usually ok)
        n4_filter.SetNumberOfFittingLevels(4)

        # Optionally, set parameters (like number of iterations per level)
        n4_filter.SetMaximumNumberOfIterations([100, 70, 50, 30]) # for an input volume of size [120, 528, 528]

    # Update the filter to perform the correction
    n4_filter.Update()

    # Get the output image
    output_itk_volume = n4_filter.GetOutput()

    return output_itk_volume


In [46]:
def numpy_to_itk(numpy_array, reference_itk_volume):

    # Convert back to ITK image (note: GetArrayFromImage returns z,y,x => shape must match!)
    img_itk = itk.GetImageFromArray(numpy_array)

    # Copy spatial metadata from input_volume_1
    img_itk.SetOrigin(reference_itk_volume.GetOrigin())
    img_itk.SetSpacing(reference_itk_volume.GetSpacing())
    img_itk.SetDirection(reference_itk_volume.GetDirection())

    return img_itk


## Dicom Information

In [47]:
def flatten_tags_dictionary(tag_dictionary):
    # TAG_DICTIONARY is a 2-level dictionary
    # The output is a 1-d list containing the 2nd level tags 

    upper_level_tags = tag_dictionary.keys()
    flattened_tags = []
    for ul_tag in upper_level_tags:
        subsetDataLabels = list(tag_dictionary[ul_tag].keys())
        flattened_tags += subsetDataLabels

    # Add the absolute filepath to the dataframe:
    flattened_tags.append('AbsFilePath')

    return flattened_tags


In [48]:
def load_dicom_file(path_to_dicom_file, tags_to_read, read_pixel_tag=False):

    # Reads an individual DICOM file and returns the metadata and image (if required)

    dicom_object = dcm.dcmread(path_to_dicom_file, stop_before_pixels = not(read_pixel_tag))

    dicom_metadata_fields = flatten_tags_dictionary(tags_to_read)
    dicom_metadata = [None]*len(dicom_metadata_fields)

    for tagSubSetName, tagSubSetFields in tags_to_read.items():
        # print(f'TagSubSetName: {tagSubSetName}')
        for tagName, tagValue in tagSubSetFields.items():
            try:
                dicom_metadata[dicom_metadata_fields.index(tagName)] = dicom_object[tagValue['hex']].value
            except Exception:
                # print(f'Attribute {tagName} does not exist, skipping it...')
                pass
    
    # Add the filepath:
    dicom_metadata[dicom_metadata_fields.index('AbsFilePath')] = path_to_dicom_file
    dicom_object = {'metadata': dicom_metadata}

    if read_pixel_tag:
        dicom_object['image'] = dicom_object.pixel_array

    return dicom_object 


In [49]:
def get_dcm_files_recursively(folder_path, file_extension="dcm"):
    """
    Recursively retrieves all DICOM files from a folder and its sub-folders.

    Parameters:
        folder_path (str): The path to the folder to search.
        file_extension (str): The file extension to look for (default is "dcm").

    Returns:
        list: A list of absolute file paths to the DICOM files.

    Example usage:
    folder_path = "/path/to/dicom/folder"
    dcm_files = get_dcm_files_recursively(folder_path)
    print(f"Found {len(dcm_files)} DICOM files.")
    for dcm_file in dcm_files:
        print(dcm_file)
    
    """
    if not os.path.isdir(folder_path):
        raise ValueError(f"The folder path '{folder_path}' does not exist or is not a directory.")

    # Use glob to search recursively for files with the specified extension
    search_pattern = os.path.join(folder_path, f"**/*.{file_extension}")
    dcm_files = glob.glob(search_pattern, recursive=True)

    return dcm_files



In [50]:
def sort_timing_dicom_dataframe(dicom_dataframe):
    # This is the main value added by reading the dicom files by ourselves:
    # Includes time attributes to plot dynamic series

    # Create datetime objects for the paired date and time attributes
    # They must be included as outer keys in TAGS_TO_READ 
    date_time_attr = ['InstanceCreation', 'Study', 'Series', 'Acquisition', 'Content']

    for datetime_tagName in date_time_attr:
        date_attr = datetime_tagName+'Date'
        time_attr = datetime_tagName+'Time'
        dicom_dataframe[datetime_tagName+'DateTime'] = parse_mixed_datetime(dicom_dataframe[date_attr] + dicom_dataframe[time_attr])

    # Additionally, merges the Study Date with Contrast Start/End to ensure they fit in the acquisition window:
    dicom_dataframe['ContrastBolusStartDateTime'] = parse_mixed_datetime(dicom_dataframe['StudyDate'] + dicom_dataframe['ContrastBolusStartTime'])
    dicom_dataframe['ContrastBolusEndDateTime'] = parse_mixed_datetime(dicom_dataframe['StudyDate'] + dicom_dataframe['ContrastBolusStopTime'])

    # Ensures PatientName is a string:
    dicom_dataframe['PatientName'] = dicom_dataframe['PatientName'].astype(str)

    # # Setup a per-slice timing:
    # # For each Series do the following:
    # #   - Sort, in ascending order, the TemporalPositionIdentifier
    # #   - Sort, in ascending order, the slice location (from - to +)
    # #   - Based on the dataframe index, loop over the Temporal position and populate the new field AcquisitionDateTimeSlice:
    # # 
    dicom_dataframe['perSliceAcquisitionTime'] = [None]*len(dicom_dataframe)
    dicom_dataframe['perSliceAcquisitionTimeInSecs'] = [None]*len(dicom_dataframe)
    dicom_dataframe['ContrastBolusStartTimeInSecs'] = [None]*len(dicom_dataframe)

    # Patient List:
    patientList = dicom_dataframe['PatientName'].unique().tolist()
    for patient in patientList:
        # Studies List for each patient:
        patient_in_df = dicom_dataframe[dicom_dataframe['PatientName'].isin([patient])]
        studies_in_patient = patient_in_df['StudyDate'].unique().tolist()
        for study_patient in studies_in_patient:
            study_in_patient = patient_in_df[patient_in_df['StudyDate'].isin([study_patient])]
            sequences_in_study = study_in_patient['SeriesDescription'].unique().tolist()
            # min acquisition time between Dyn eTHRIVE and 4D_THRIVE:
            df_aux = study_in_patient[(study_in_patient['SeriesDescription'].str.startswith('4D_THRIVE')) | (study_in_patient['SeriesDescription'].str.startswith('Dyn'))]
            min_time = df_aux['AcquisitionDateTime'].min(skipna=True)
            for sequence_study_patient in sequences_in_study:
                # We're now at the level of series.
                # For the timming, we're only interested in 4D_THRIVE_Ultrafast & Dyn eTHRIVE:
                if sequence_study_patient.startswith('4D_THRIVE') | sequence_study_patient.startswith('Dyn'):
                    serie_in_study = study_in_patient[study_in_patient['SeriesDescription'].isin([sequence_study_patient])]
                    # Additional Filter to remove the corrupted single file:
                    serie_in_study = serie_in_study[serie_in_study['AcquisitionNumber'].notna()]
                    # Now, we should be ready to assign the timings:
                    # Sort by temporal position:
                    serie_in_study.sort_values(by=['TemporalPositionIdentifier', 'SliceLocation'], ascending=(True, False), inplace=True)
                    nslices = len(serie_in_study['SliceLocation'].unique().tolist())
                    slice_order = range(0, nslices)
                    # Here loop over TimePos and slice location using the formula:
                    # AcquisitionTime  + TR * TempPos * Range(Slice)
                    TR = serie_in_study['RepetitionTime'].unique().tolist()
                    if len(TR) > 1:
                        print(f'WARNING!: There are more than 1 TR ({TR}) in the series. Using the first one available')
                    # else:
                        # print(f'TR: {TR[0]}ms')
                    TR = TR[0]

                    nt = serie_in_study['NumberOfTemporalPositions'].unique().tolist()
                    if len(nt) > 1:
                        print(f'ERROR: The number of temporal positions {nt} is not consistent through the series')
                        sys.exit()
                    # else:
                        # print(f'Number of Temporal Positions: {nt[0]}')                
                    nt = int(nt[0])
                    slices_positions = list(slice_order)*nt
                    perSliceAcquisitionDateTime = pd.to_timedelta([z*TR/1000.0 for z in slices_positions], unit='s')
                    # serie_in_study['perSliceAcquisitionDateTime'] = [None]*len(serie_in_study)
                    serie_in_study['perSliceAcquisitionDateTime'] = serie_in_study['AcquisitionDateTime'] + perSliceAcquisitionDateTime
                    
                    # Find the earliest AcquisitionEndTime (ignoring NaT values) -- But the min time in my case, is the start of the Dyn eTHRIVE acquisition
                    # min_time = serie_in_study['perSliceAcquisitionDateTime'].min(skipna=True)
                    # Compute time differences in seconds for the acquisition and contrast bolus
                    serie_in_study['perSliceAcquisitionTimeInSecs'] = (serie_in_study['perSliceAcquisitionDateTime'] - min_time).dt.total_seconds()
                    serie_in_study['ContrastBolusStartTimeInSecs'] = (serie_in_study['ContrastBolusStartDateTime'] - min_time).dt.total_seconds()

                    dicom_dataframe.loc[serie_in_study.index, 'perSliceAcquisitionDateTime'] = serie_in_study['perSliceAcquisitionDateTime']
                    dicom_dataframe.loc[serie_in_study.index, 'perSliceAcquisitionTimeInSecs'] = serie_in_study['perSliceAcquisitionTimeInSecs']
                    dicom_dataframe.loc[serie_in_study.index, 'ContrastBolusStartTimeInSecs'] = serie_in_study['ContrastBolusStartTimeInSecs']
                    
    # Add time axis in minutes too:
    dicom_dataframe[['perSliceAcquisitionTimeInMins', 'ContrastBolusStartTimeInMins']] = dicom_dataframe[['perSliceAcquisitionTimeInSecs', 'ContrastBolusStartTimeInSecs']]/60.0

    # Slicer3D uses the image position patient coordinate, rather than SliceLocation:
    dicom_dataframe[['x','y','z']] = dicom_dataframe['ImagePositionPatient'].apply(pd.Series)
    
    return dicom_dataframe



In [51]:
def get_image_dataset(dataset_df, use_philips_rescale=True):
    """ 
    This function loads the image corresponding a single dataset. 
    I define a single dataset as a set of dicom files within a single sequence (i.e. SeriesDescription is a single value) 
    It requires the image's height and width are the same for all the elements in the dataset. 
    It should be able to deal with 3D (volumetric) and dynamic (time) series, either 2D+time or 4D (3D + time)

    Args:
        dataset_df (_type_): _description_
        use_philips_rescale (bool, optional): _description_. Defaults to True.

    Returns:
        _type_: _description_
    """
    # Ensure the dataset is a single dataset:
    patient_name = dataset_df['PatientID'].unique().tolist()
    if len(patient_name) > 1:
        print(f'WARNING: The dataset contains more than one patient ({patient_name}), check the input')
        sys.exit()
    patientID = patient_name[0]

    scan_date = dataset_df['StudyDate'].unique().tolist()
    if len(scan_date) > 1:
        print(f'WARNING: The dataset contains more than one study date ({scan_date}), check the input')
        sys.exit()
    scan_date = scan_date[0]

    sequence_name = dataset_df['SeriesDescription'].unique().tolist()
    if len(sequence_name) > 1:
        print(f'WARNING: The dataset contains more than one sequence ({sequence_name}), check the input')
        sys.exit()
    sequence_name = sequence_name[0]

    # Rows and Columns must be the same for the whole series:
    nrows_list = dataset_df['Rows'].unique()
    ncols_list = dataset_df['Columns'].unique()
    if (len(nrows_list) > 1) | (len(ncols_list)>1):
        print(f'WARNING: Either the number of ROWS ({nrows_list}) or COLUMNS ({ncols_list}) is not consistent for the Series')
    [w, h] = [ncols_list[0], nrows_list[0]]

    slice_locations = dataset_df['z'].unique().tolist()
    slice_locations.sort(reverse=False)
    
    ns = len(slice_locations)

    number_of_temporal_positions = dataset_df['NumberOfTemporalPositions'].unique()
    # numberOfTempPositions must be only 1 element length, because it must be the same for a single Series (afaik)
    temporal_positions = dataset_df['TemporalPositionIdentifier'].unique().tolist()
    temporal_positions.sort(reverse=False)

    
    if len(number_of_temporal_positions) > 1:
        print(f'WARNING: Check you are processing only one series, the length of temporal_positions is {len(number_of_temporal_positions)}')
        sys.exit()
    nt = number_of_temporal_positions[0]

    print(f'Dataset {patientID}-{scan_date}-{sequence_name} contains {ns} slices and {nt} temporal positions of size {w}x{h} (WxH)')

    # Image resolution and slice spacing
    # df_metadata_patient_visit_sequence['PixelSpacing']
    [resRows, resCols]= dataset_df['PixelSpacing'].apply(pd.Series).T.values.tolist()
    deltaRow, deltaCol = [np.unique(resRows), np.unique(resCols)]

    # resolution along the slice direction:
    # delta_z = dataset_df['SliceThickness'].unique()
    delta_z = dataset_df['SpacingBetweenSlices'].unique()

    if (len(deltaRow) > 1) | (len(deltaCol) > 1) | (len(delta_z) > 1):
        print('WARNING: Check you are processing only one series')
        print(f'\t the length of the resolutions parameters is not 1 for Rows ({len(deltaRow)}), Columns ({len(deltaCol)}) and/or slice ({len(delta_z)})')
        sys.exit()

    resx_W, resy_H, resz_Z = [deltaRow[0], deltaCol[0], delta_z[0]]
    print(f'Image resolution is {resx_W:.3f}x{resy_H:.3f}x{resz_Z:.2f} [mm/pixel] (WxHxZ)')


    # Load the data, remember the dataframe is already sorted by slice->time:
    # Remember the array order in numpy is W(rows), H(cols), Z(slice)
    image4D = np.full((nt, ns, w, h), np.nan)
    timeaxis_spatial_temporal = np.full((ns, nt), np.nan)
    zlocation_spatial_temporal = np.full((ns, nt), np.nan)

    for dicom_file_row, dicom_datafile in dataset_df.iterrows():
        dcm_object = dcm.dcmread(dicom_datafile['AbsFilePath'])
        slice_position = slice_locations.index(dicom_datafile["z"])
        time_position = temporal_positions.index(dicom_datafile["TemporalPositionIdentifier"])
        # print(f'Slice Position: {slice_position}')
        # print(f'Temporal Position: {time_position}')
        
        dcm_image_32bit = dcm_object.pixel_array.astype(np.float32)
        # if rescale:
        # According to this thread https://stackoverflow.com/questions/67889762/what-is-the-difference-between-rescale-slope-intercept-and-scale-slope-inter
        # The forumalea relevant are:
        # D = R * RS + RI; R= raw pixel value, RS Rescale Slope (0028,1053), RI Rescale Intercept (0028,1052)
        # P = D / (RS * SS); D= Displayed Value (in the scanner screen), RS Rescale Slope (0028,1053), SS Scale Slope (2005,100E) "PhilipsScaleSlope"
        dcm_image_32bit *= dicom_datafile['RescaleSlope']
        dcm_image_32bit += dicom_datafile['RescaleIntercept']
        if use_philips_rescale:
            dcm_image_32bit /= (dicom_datafile['RescaleSlope'] * dicom_datafile['PhilipsScaleSlope'])

        # image4D[:,:,slice_position, time_position] = dcm_image_32bit
        image4D[time_position, slice_position, :, :] = dcm_image_32bit
        # Get the timing (secs) and location (mm) of each slice, in secs. It allows to easily get the time axis when plotting the data
        timeaxis_spatial_temporal[slice_position, time_position] = dicom_datafile['perSliceAcquisitionTimeInSecs']
        zlocation_spatial_temporal[slice_position, time_position] = dicom_datafile['z']
    

    image4D = np.squeeze(image4D)
    scanDetails = {'scanDateTime': dataset_df,
                   'sequenceName': dataset_df,
                   'spacing': [resx_W, resy_H, resz_Z],
                   'TR': dataset_df['RepetitionTime'].unique().tolist(),
                   'TE': dataset_df['EchoTime'].unique().tolist(),
                   'FA': dataset_df['FlipAngle'].unique().tolist(),
                   'ETL': dataset_df['EchoTrainLength'].unique().tolist()
                   }

    return image4D, zlocation_spatial_temporal, timeaxis_spatial_temporal, scanDetails


In [52]:
# Loop over a folder with DICOM files (they may be inside subfolders too) and create a dataframe with the metadata of all the dicom files identified
def load_dicom_folder(path_to_dicom_folder, tags_to_read, fext='dcm', read_pixel_tag=False):
    # Reads a folder with DICOM files and returns a dataframe with the metadata and images (if required)
    # check the folder recursiverly for DICOM files
    dcmlist = get_dcm_files_recursively(path_to_dicom_folder, fext)
    print(f'There are {len(dcmlist)} files in {path_to_dicom_folder}')    

    dicom_metadata_hdr = flatten_tags_dictionary(tags_to_read)
    dicomMetadataList = []
    for dcmfile in dcmlist:
        dcm_metadata = load_dicom_file(dcmfile, tags_to_read, read_pixel_tag)
        dicomMetadataList.append(dcm_metadata['metadata'])

    dicom_dataframe = pd.DataFrame(columns=dicom_metadata_hdr, data=dicomMetadataList)#.astype(dtype_df_dict)

    dicom_dataframe = sort_timing_dicom_dataframe(dicom_dataframe)
    
    return dicom_dataframe


In [53]:
dicom_dictionary = {
'ID': { # Patient and Study ID
            'StudyID': {'hex': 0x00200010,
                         #  'type': str
                         },
            'PatientName': {'hex': 0x00100010,
                         #    'type': str
                            },
            'PatientID': {'hex': 0x00100020,
                         #  'type': str
                         }
     },
'SeriesID': { # Acquisitions and Series ID
            'SeriesDescription': {'hex': 0x0008103e,
                         #  'type': str
                         },
            'SeriesNumber': {'hex': 0x00200011,
                         #  'type': pd.Int64Dtype()
                         },
            'AcquisitionNumber': {'hex': 0x00200012,
                              #     'type': pd.Int64Dtype()
                                  },
            'InstanceNumber':{'hex': 0x00200013,
                              # 'type': pd.Int64Dtype()
                             }
            # 'ImagesInAcquisition': {'hex': 0x00201002,
            #                   #     'type': pd.Int64Dtype()
            #                       }
          },
'DatesTimes': { #  Dates & Times
            'StudyDate': {'hex': 0x00080020,
                         #  'type': str
                         },
            'StudyTime': {'hex': 0x00080030,
                         #  'type': str
                         },
            'SeriesDate': {'hex': 0x00080021,
                         #  'type': str
                         },
            'SeriesTime': {'hex': 0x00080031,
                         #  'type': str
                         },
            'AcquisitionDate': {'hex': 0x00080022,
                         #  'type': str
                         },
            'AcquisitionTime': {'hex': 0x00080032,
                         #  'type': str
                         },
          #   'AcquisitionDateTime': {'hex': 0x0008002A,
          #                          'type': str
          #                },
            'AcquisitionDuration': {'hex': 0x00189073, # Duration of the single continuous gathering of data over a period of time that resulted in this instance, in seconds.
                                   #  'type': pd.Float64Dtype()
                                    },
            'ContentDate': {'hex': 0x00080023,
                         #  'type': str
                         },
            'ContentTime': {'hex': 0x00080033,
                         #    'type': str
                            },
            'InstanceCreationDate': {'hex': 0x00080012,
                         #  'type': str
                         },
            'InstanceCreationTime': {'hex': 0x00080013,
                         #  'type': str
                         },
            'PerformedProcedureStepEndDate': {'hex': 0x00400250,
                         #  'type': str
                         },
            'PerformedProcedureStepEndTime': {'hex': 0x00400251,
                         #  'type': str
                         }
          },
'AcqPars': { # Acquisition Parameters
            'ImageType': {'hex': 0x00080008,
                        #   'type': 
                        },
            'ScanningSequence': {'hex': 0x00180020,
                                 },
            'SequenceVariant': {'hex': 0x00180021,
                                 },
            'ScanOptions': {'hex': 0x00180022, # Parameters of scanning sequence.
                         #    'type': str
                            },
            'MRAcquisitionType': {'hex': 0x00180023,
                              #     'type': str
                                  },
            'SequenceName': {'hex': 0x00180024, 
                         #    'type': str
                            },
            'RepetitionTime': {'hex': 0x00180080,
                              #  'type': pd.Float64Dtype()
                               },
            'EchoTime': {'hex': 0x00180081,
                         # 'type': pd.Float64Dtype()
                         },
            'FlipAngle': {'hex': 0x00181314,
                         #  'type': pd.Float64Dtype()
                          },
            'InversionTime': {'hex': 0x00180082,
                              # 'type': pd.Float64Dtype()
                         },
            'NumberOfAverages': {'hex': 0x00180083,
                              # 'type': pd.Int64Dtype()
                         },
            'NumberOfPhaseEncodingSteps': {'hex': 0x00180089,
                                        #    'type': pd.Int64Dtype()
                                           },
            'EchoTrainLength': {'hex': 0x00180091,
                              #   'type': pd.Int64Dtype()
                                },
            'PercentSampling': {'hex': 0x00180093,
                              #   'type': pd.Float64Dtype()
                                },
            'PercentPhaseFieldOfView': {'hex': 0x00180094,
                                        # 'type': pd.Float64Dtype()
                                        },
            'PixelBandwidth': {'hex': 0x00180095,
                              #  'type': pd.Float64Dtype()
                               },
            'TriggerTime': {'hex': 0x00181060, # Time, in msec, between peak of the R wave and the peak of the echo produced
                         #    'type': pd.Float64Dtype()
                            },
            'InPlanePhaseEncodingDirection': {'hex': 0x00181312,
               #   'type': str
                 },
            'TemporalPositionIdentifier': {'hex': 0x00200100,
               #   'type': str
                 },
            'NumberOfTemporalPositions': {'hex': 0x00200105,
                                        #   'type': str
                                          },
            # 'TemporalResolution': {'hex': 0x00200110,
            #                             #   'type': str
            #                               },
            'SliceLocation':{'hex': 0x00201041,
                         #     'type': pd.Float64Dtype()
                            },
            'ImagePositionPatient':{'hex': 0x00200032,
                                   #  'type': object
                                    },
            'SliceThickness': {'hex': 0x00180050,
                              #  'type': pd.Float64Dtype()
                               },
            'SpacingBetweenSlices': {'hex': 0x00180088,
                                   #   'type': pd.Float64Dtype()
                              },
            'ImageOrientationPatient': {'hex': 0x00200037,
                                        # 'type': object
                              },
            'IsoCenterPosition': {'hex': 0x300A012C,
                              }
          },
'ImagePars': { # Reconstruction/Image Parameters
            'AcquisitionMatrix': {'hex': 0x00181310,
                              #     'type': object
                                  },
            'PixelSpacing': {'hex': 0x00280030,
                         #     'type': object
                             },
            'Rows': {'hex': 0x00280010,
                    #  'type': pd.Int64Dtype()
                     },
            'Columns': {'hex': 0x00280011,
                    #     'type': pd.Int64Dtype()
                        },
            # 'PixelAspectRatio': {'hex': 0x00280034,
            #                   #    'type': object
            #                      },
            'ReconstructionDiameter': {'hex': 0x00181100,
                                   #     'type': pd.Float64Dtype()
                                       },
            'RescaleType': {'hex': 0x00281054,
                         #    'type': str
                            },
            'RescaleIntercept': {'hex': 0x00281052, # RI
                         #    'type': pd.Float64Dtype()
                            },
            'RescaleSlope': {'hex': 0x00281053, # RS
                         #    'type': pd.Float64Dtype()
                            },
            # 'PhilipsRWVSlope': {'hex': 0x00409225,  # Philips Private Attribute
            #                   #   'type': pd.Float64Dtype()
            #                    },
            # 'PhilipsRWVIntercept': {'hex': 0x00409224,  # WI, Philips Private Attribute
            #                        #  'type': pd.Float64Dtype()
            #                         },
            'PhilipsScaleSlope': {'hex': 0x2005100E, # SS, Philips Private Attribute
                              #     'type': pd.Float64Dtype()
                                  }
            # 'PhaseNumber': {'hex': 0x20011008, # Philips Private Attribute
            #              #    'type': pd.Int64Dtype()
            #                 }
          },
'ContrastAgent': { # Contrast Agent
            'ContrastBolusAgent': {'hex': 0x00180010,
                         #  'type': str
                         },
            'ContrastBolusRoute': {'hex': 0x00181040,
                         #  'type': str
                         },
            'ContrastBolusVolume': {'hex': 0x00181041,
                         #  'type': pd.Float64Dtype()
                         },
            'ContrastBolusStartTime': {'hex': 0x00181042,
                         #  'type': pd.Float64Dtype()
                         },
            'ContrastBolusStopTime': {'hex': 0x00181043,
                         #  'type': pd.Float64Dtype()
                         },
            'ContrastBolusTotalDose': {'hex': 0x00181044,
                         #  'type': pd.Float64Dtype()
                         },
            'ContrastFlowRate': {'hex': 0x00181046,
                         #  'type': pd.Float64Dtype()
                         },
          }
   }


# Paths and environment variables

In [54]:
HOMEPATH = getenv()
DATAPATH = os.path.join(HOMEPATH, 'Data', 'FTV_DCEMRI_Phase02')

DCMPATH = os.path.join(DATAPATH, 'DICOM')
REGPATH = os.path.join(DATAPATH, 'registration')
T1PATH = os.path.join(DATAPATH, 't1vfa')
TMPPATH = os.path.join(DATAPATH, 'tmp')
PRELIMPATH = os.path.join(DATAPATH, 'preliminary_review')

FEXT = 'dcm'

sub_folder_list = [DCMPATH, REGPATH, T1PATH, TMPPATH, PRELIMPATH]
# Check the sub-folder exist, if not creates it:
for sub_folder in sub_folder_list:
    os.makedirs(sub_folder, exist_ok=True)
    
patient_list = list_folder(DCMPATH, sorted=True)


# Registration Parameters
Define a parameters object using the parameters files available in the [Model Zoo](https://lkeb.ml/modelzoo/). For breast MRI, can use some of the cited in the literature:
* Par0032: [Description](https://lkeb.ml/modelzoo/par0032/) - [Parameter files](https://github.com/SuperElastix/ElastixModelZoo/tree/master/models/Par0032) 
* Par0052: [Description](https://lkeb.ml/modelzoo/par0052/) - [Parameter files](https://github.com/SuperElastix/ElastixModelZoo/tree/master/models/Par0052) 
* Par0058: [Description](https://lkeb.ml/modelzoo/par0058/) **(not yey used)** - [Parameter files](https://github.com/SuperElastix/ElastixModelZoo/tree/master/models/Par0058) 

In [58]:
# Create parameters object for a given set of parameters:
parameter_object = itk.ParameterObject.New()

# Where to save the parameters files
reg_pars_save_path = os.path.join(REGPATH, 'pars')
os.makedirs(reg_pars_save_path, exist_ok=True)

url_trunk = 'https://raw.githubusercontent.com/SuperElastix/ElastixModelZoo/refs/heads/master/models/'
parametersID = 'Par0032'
parametersFile = {'Par0032': ['Par0032_bsplines.txt', 'Par0032_rigid.txt'],
                  'Par0052': ['Elastix_Params_Affine.txt', 'Elastix_Params_BSpline.txt'],
                  'Par0058': ['Par0058trans.txt']
}

for parameters_filename in parametersFile[parametersID]:
    url_to_parameter_file = os.path.join(url_trunk, parametersID, parameters_filename)
    local_path_file = os.path.join(reg_pars_save_path, parameters_filename)

    registration_parameter_file_obj = urlopen(url_to_parameter_file)
    with open(local_path_file, "wb") as f:
        f.write(registration_parameter_file_obj.read())

    if os.path.isfile(local_path_file):
        parameter_object.AddParameterFile(local_path_file)

print(parameter_object)


ParameterObject (0x3027d4f40)
  RTTI typeinfo:   elastix::ParameterObject
  Reference Count: 1
  Modified Time: 589478
  Debug: Off
  Object Name: 
  Observers: 
    none
ParameterMap 0: 
  (BSplineInterpolationOrder 1)
  (CompressResultImage "true")
  (DefaultPixelValue 0)
  (ErodeMask "false")
  (FinalBSplineInterpolationOrder 1)
  (FinalGridSpacingInPhysicalUnits 40)
  (FixedImageDimension 3)
  (FixedImagePyramid "FixedRecursiveImagePyramid")
  (FixedInternalImagePixelType "short")
  (HowToCombineTransforms "Compose")
  (ImagePyramidSchedule 4 4 4 2 2 2 1 1 1)
  (ImageSampler "Random")
  (Interpolator "BSplineInterpolator")
  (MaximumNumberOfIterations 500)
  (Metric "AdvancedMattesMutualInformation")
  (MovingImageDimension 3)
  (MovingImagePyramid "MovingRecursiveImagePyramid")
  (MovingInternalImagePixelType "short")
  (NewSamplesEveryIteration "true")
  (NumberOfHistogramBins 32)
  (NumberOfResolutions 3)
  (NumberOfSpatialSamples 5000)
  (Optimizer "AdaptiveStochasticGradientDe

# Get DICOM metadata for the whole study
This may take longer (around 1-2min) but will get all datasets in memory, so can easily switch between patients


In [59]:
# Get the list of files for the whole study:
dicom_dataframe = load_dicom_folder(DCMPATH, dicom_dictionary, fext=FEXT)
dicom_dataframe.head()

# This is very particular to how the dataset is organised, the patientID numbers are in the folder names
# Just for simplicity (and human readibility), construct a dictionary between the ID numbers and actual ID
# but for the code point of view, the actual ID is the relevant selector:
patientsID = dicom_dataframe['PatientID'].unique().tolist()
patientsIDnro = [dicom_dataframe['AbsFilePath'].loc[dicom_dataframe['PatientID']==patientID].str.slice(len(DCMPATH)+1, len(DCMPATH)+4).unique().tolist()[0] for patientID in patientsID]
patientsID = dict(sorted(dict(zip(patientsIDnro, patientsID)).items()))
patientsID


There are 58080 files in /Users/joseulloa/Data/FTV_DCEMRI_Phase02/DICOM


{'001': 'ANON58704',
 '002': 'ANON67302',
 '004': 'ANON44148',
 '005': 'ANON93295',
 '006': 'ANON70393',
 '008': 'ANON10181',
 '009': 'ANON89649'}

# Pre-Processing steps

In [73]:
# Noise Reduction parameters:
noiseRedMethod = 'gaussian' # None, 'gaussian' or 'median'
# Median Filter: Define the radius of the neighborhood
radius = 1
# Gaussian Filter
gvar = 2.0

# for the 1D plots, will average over a 2*roiWidth pixels box
roiWidth = 5 

noiseRedFiltFlag = False
biasFlag = False
regFlag  = False
thirdFAflag = False
smoothFlag = False

usePhilipsReScale = False
loadMethod = 'dcm' # 'itk' or 'dcm'


# Select Dataset


In [ ]:
patientIDnro = '009'
visitNro = 1 # 1 or 2

patientID = patientsID[patientIDnro]
# Filter the main dataframe to get the patient of interest:
patient_in_df = dicom_dataframe[dicom_dataframe['PatientID'].isin([patientID])]

# For the selected patient, gets the study dates. There are up to 2 visits per patient, the earliest must be the visit 1:
visitsDates = sorted(patient_in_df['StudyDate'].unique().tolist())
visitsID = dict(zip([1, 2],visitsDates))
visitID = visitsID[visitNro]
studyDateFMT = dtime.strftime(dtime.strptime(visitID, "%Y%m%d"), "%d/%m/%Y")


patient_visit_df = patient_in_df[patient_in_df['StudyDate'].isin([visitID])]

patient_visit_df["StudyID"].unique().tolist()[0]
print('Patient Metadata:')
print(f'\tStudy ID: {patient_visit_df["StudyID"].unique().tolist()[0]} ({visitID})')
print(f'\tPatient ID: {patient_visit_df["PatientID"].unique().tolist()[0]} ({patientID}) ')
print(f'\tStudyDate: {studyDateFMT}')

# Get the list of sequences for the patient and visit of interest:
sequences_in_visit = patient_visit_df[patient_visit_df['StudyDate'].isin([visitID])]['SeriesDescription'].unique().tolist()


Patient Metadata:
	Study ID: ANON10181 (20230831)
	Patient ID: ANON10181 (ANON10181) 
	StudyDate: 31/08/2023


### Loading imaging parameters

In [278]:
start_time = time.perf_counter()
accrued_time = 0
map_filename_pattern = f'{patientIDnro}_v{visitNro}_{patientID}_{visitID}_{loadMethod}'
if usePhilipsReScale & (loadMethod == 'dcm'):
    map_filename_pattern += '_phlpScale'
print(''.join(['§']*50))
print('Get Sequence parameters')
## VFA FA=15
fixed_sequence_pattern = 'FA15'
fixed_sequence_df = patient_visit_df[patient_visit_df['AbsFilePath'].str.contains(fixed_sequence_pattern)]
fixed_sequenceID = fixed_sequence_df['AbsFilePath'].apply(os.path.dirname).unique().tolist()[0]
fixed_flip_angle = fixed_sequence_df['FlipAngle'].unique().tolist()
fixed_TR = fixed_sequence_df['RepetitionTime'].unique().tolist()

if len(fixed_flip_angle) > 1:
    print(f'WARNING: The dataset contains more than one flip angle ({fixed_flip_angle}), check the input')
    sys.exit()
fixed_flip_angle = fixed_flip_angle[0]
print(f'Fixed FA: {fixed_flip_angle}deg')

if len(fixed_TR) > 1:
    print(f'WARNING: The dataset contains more than one TR ({fixed_TR}), check the input')
    sys.exit()
fixed_TR = fixed_TR[0]
print(f'Fixed TR: {fixed_TR}ms')

## VFA FA=5
moving_sequence_pattern = 'FA5'
moving_sequence_df = patient_visit_df[patient_visit_df['AbsFilePath'].str.contains(moving_sequence_pattern)]
moving_sequenceID = moving_sequence_df['AbsFilePath'].apply(os.path.dirname).unique().tolist()[0]
moving_flip_angle = moving_sequence_df['FlipAngle'].unique().tolist()
moving_TR = moving_sequence_df['RepetitionTime'].unique().tolist()

if len(moving_flip_angle) > 1:
    print(f'WARNING: The moving sequence contains more than one flip angle ({moving_flip_angle}), check the input')
    sys.exit()
moving_flip_angle = moving_flip_angle[0]
print(f'Moving FA: {moving_flip_angle}deg')

if len(moving_TR) > 1:
    print(f'WARNING: The dataset contains more than one TR ({moving_TR}), check the input')
    sys.exit()
moving_TR = moving_TR[0]
print(f'Moving TR: {moving_TR}ms')

## Dynamic FA=10
dynamic_sequence_pattern = '4D'
dynamic_sequence_df = patient_visit_df[patient_visit_df['AbsFilePath'].str.contains(dynamic_sequence_pattern)]
dynamic_sequenceID = dynamic_sequence_df['AbsFilePath'].apply(os.path.dirname).unique().tolist()[0]
dynamic_flip_angle = dynamic_sequence_df['FlipAngle'].unique().tolist()
dynamic_TR = dynamic_sequence_df['RepetitionTime'].unique().tolist()

if len(dynamic_flip_angle) > 1:
    print(f'WARNING: The moving sequence contains more than one flip angle ({dynamic_flip_angle}), check the input')
    sys.exit()
dynamic_flip_angle = dynamic_flip_angle[0]
print(f'Moving FA: {dynamic_flip_angle}deg')

if len(dynamic_TR) > 1:
    print(f'WARNING: The dataset contains more than one TR ({dynamic_TR}), check the input')
    sys.exit()
dynamic_TR = dynamic_TR[0]
print(f'Moving TR: {dynamic_TR}ms')
print(''.join(['§']*50))

# Look at how reliable would be the TR:
TRlist = [fixed_TR, moving_TR, dynamic_TR]
meanTR, stdTR = [np.mean(TRlist), np.std(TRlist)]
print(f'TR for each Flip Angle: {TRlist}')
print(f'\tTR STDev: {stdTR:.2f}')
print(f'\tmean TR: {meanTR:.2f}ms')
if stdTR > 0.1: # TODO: Refine this threshold
    print(f'[WARNING]: TR may be too different among the volumes: {TRlist}')

print(''.join(['§']*50))
end_time = time.perf_counter()
elp_time = end_time - start_time
accrued_time += elp_time
print(f'§§§§ Setting up data took {elp_time:.2f}[s]')
print(''.join(['§']*50))


§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§
Get Sequence parameters
Fixed FA: 15.0deg
Fixed TR: 5.18240022659301ms
Moving FA: 5.0deg
Moving TR: 5.18240022659301ms
Moving FA: 10.0deg
Moving TR: 5.18240022659301ms
§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§
TR for each Flip Angle: [5.18240022659301, 5.18240022659301, 5.18240022659301]
	TR STDev: 0.00
	mean TR: 5.18ms
§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§
§§§§ Setting up data took 0.01[s]
§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§


### Load the DICOM files

In [279]:
start_time = time.perf_counter()

print('Load the DICOM files')
## VFA FA=15
fixed_volume_dcm, fixed_slice_position_matrix, fixed_abs_time_matrix, fixed_scanDetails = get_image_dataset(fixed_sequence_df, 
                                                                                                        use_philips_rescale=usePhilipsReScale)
fixed_avg_dicom = np.mean(fixed_volume_dcm)
print(f'Average Image intensity (DICOM): {fixed_avg_dicom}')

## VFA FA=5
moving_volume_dcm, moving_slice_position_matrix, moving_abs_time_matrix, moving_scanDetails = get_image_dataset(moving_sequence_df, 
                                                                                                            use_philips_rescale=usePhilipsReScale)
moving_avg_dicom = np.mean(moving_volume_dcm)
print(f'Average Image intensity (DICOM): {moving_avg_dicom}')

## Dynamic FA=10
dynamic_volume_dcm, dynamic_slice_position_matrix, dynamic_abs_time_matrix, dynamic_scanDetails = get_image_dataset(dynamic_sequence_df, 
                                                                                                                use_philips_rescale=usePhilipsReScale)
dynamic_avg_dicom = np.mean(dynamic_volume_dcm)
print(f'Average Image intensity (DICOM): {dynamic_avg_dicom}')
[nt, ns, w, h] = dynamic_volume_dcm.shape
moving_10deg_volume_dcm = dynamic_volume_dcm[0, :, :, :]
print(''.join(['§']*50))

print('Creating ITK objects from DICOM arrays')
eye3d = np.eye(3)
eye3d[:, :2] = np.array(fixed_sequence_df['ImageOrientationPatient'].values[0]).reshape(2,3).T
fixed_direction_matrix = itk.matrix_from_array(eye3d)
fixed_spacing_vector = np.array(fixed_scanDetails['spacing'])
fixed_origin_vector = np.array(fixed_sequence_df['ImagePositionPatient'].apply(pd.Series).min())

fixed_volume_dcm2itk = itk.GetImageFromArray(fixed_volume_dcm)
fixed_volume_dcm2itk.SetSpacing(fixed_spacing_vector)
fixed_volume_dcm2itk.SetOrigin(fixed_origin_vector)
fixed_volume_dcm2itk.SetDirection(fixed_direction_matrix)
fixed_volume_dcm2itk_metadata = dict(fixed_volume_dcm2itk)
print('Image Parameters Fixed Volume (dcm2itk): ')
print(f'\tITK Origin: {fixed_volume_dcm2itk_metadata["origin"]}')
print(f'\tITK Spacing: {fixed_volume_dcm2itk_metadata["spacing"]}')
print(f'\tITK Direction: {fixed_volume_dcm2itk_metadata["direction"]}')

eye3d[:, :2] = np.array(moving_sequence_df['ImageOrientationPatient'].values[0]).reshape(2,3).T
moving_direction_matrix = itk.matrix_from_array(eye3d)
moving_spacing_vector = np.array(moving_scanDetails['spacing'])
moving_origin_vector = np.array(moving_sequence_df['ImagePositionPatient'].apply(pd.Series).min())

moving_volume_dcm2itk = itk.GetImageFromArray(moving_volume_dcm)
moving_volume_dcm2itk.SetSpacing(moving_spacing_vector)
moving_volume_dcm2itk.SetOrigin(moving_origin_vector)
moving_volume_dcm2itk.SetDirection(moving_direction_matrix)
moving_volume_dcm2itk_metadata = dict(moving_volume_dcm2itk)
print('Image Parameters Moving Volume (dcm2itk): ')
print(f'\tITK Origin: {moving_volume_dcm2itk_metadata["origin"]}')
print(f'\tITK Spacing: {moving_volume_dcm2itk_metadata["spacing"]}')
print(f'\tITK Direction: {moving_volume_dcm2itk_metadata["direction"]}')

eye3d[:, :2] = np.array(dynamic_sequence_df['ImageOrientationPatient'].values[0]).reshape(2,3).T
moving_10deg_direction_matrix = itk.matrix_from_array(eye3d)
moving_10deg_spacing_vector = np.array(dynamic_scanDetails['spacing'])
moving_10deg_origin_vector = np.array(dynamic_sequence_df['ImagePositionPatient'].apply(pd.Series).min())

moving_10deg_volume_dcm2itk = itk.GetImageFromArray(moving_10deg_volume_dcm)
moving_10deg_volume_dcm2itk.SetSpacing(moving_spacing_vector)
moving_10deg_volume_dcm2itk.SetOrigin(moving_origin_vector)
moving_10deg_volume_dcm2itk.SetDirection(moving_direction_matrix)
moving_10deg_volume_dcm2itk_metadata = dict(moving_10deg_volume_dcm2itk)
print('Image Parameters Moving (from Dyn) Volume (dcm2itk): ')
print(f'\tITK Origin: {moving_10deg_volume_dcm2itk_metadata["origin"]}')
print(f'\tITK Spacing: {moving_10deg_volume_dcm2itk_metadata["spacing"]}')
print(f'\tITK Direction: {moving_10deg_volume_dcm2itk_metadata["direction"]}')

eye4d = np.eye(4)
eye4d[:-1, :2] = np.array(dynamic_sequence_df['ImageOrientationPatient'].values[0]).reshape(2,3).T
dynamic_direction_matrix = itk.matrix_from_array(eye4d)

dyn_mode, c = np.unique(np.diff(dynamic_abs_time_matrix[1,:]), return_counts=True)
time_spacing = dyn_mode[np.argmax(c)]
dynamic_spacing_vector = np.array(dynamic_scanDetails['spacing'] + [time_spacing])

dynamic_origin_vector = np.append(np.array(dynamic_sequence_df['ImagePositionPatient'].apply(pd.Series).min()), 0)
dynamic_volume_dcm2itk = itk.GetImageFromArray(dynamic_volume_dcm)
dynamic_volume_dcm2itk.SetSpacing(dynamic_spacing_vector)
dynamic_volume_dcm2itk.SetOrigin(dynamic_origin_vector)
dynamic_volume_dcm2itk.SetDirection(dynamic_direction_matrix)
print('********* Sequences loaded and converted to numpy arrays ***************')

print(''.join(['§']*50))
end_time = time.perf_counter()
elp_time = end_time - start_time
accrued_time += elp_time
print(f'§§§§ Loading data (DCM) took {elp_time:.2f}[s]')
print(''.join(['§']*50))


Load the DICOM files
Dataset ANON10181-20230831-FA15_THRIVE_map contains 120 slices and 1 temporal positions of size 528x528 (WxH)
Image resolution is 0.680x0.680x1.50 [mm/pixel] (WxHxZ)
Average Image intensity (DICOM): 261.2561887834742
Dataset ANON10181-20230831-FA5_THRIVE_map contains 120 slices and 1 temporal positions of size 528x528 (WxH)
Image resolution is 0.680x0.680x1.50 [mm/pixel] (WxHxZ)
Average Image intensity (DICOM): 245.40192686744248
Dataset ANON10181-20230831-4D_THRIVE_Ultrafast contains 120 slices and 28 temporal positions of size 528x528 (WxH)
Image resolution is 0.680x0.680x1.50 [mm/pixel] (WxHxZ)
Average Image intensity (DICOM): 359.28168210500735
§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§§
Creating ITK objects from DICOM arrays
Image Parameters Fixed Volume (dcm2itk): 
	ITK Origin: [ -80.86397552  -92.27231911 -179.24147975]
	ITK Spacing: [1.5        0.67961162 0.67961162]
	ITK Direction: [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
Image Parameters Moving Volume 

### Load the data using ITK

In [ ]:
start_time = time.perf_counter()

print('Load the data using ITK')

## VFA FA=15
fixed_volume_itk = itk.imread(fixed_sequenceID, itk.F)
fixed_volume_itk_metadata = dict(fixed_volume_itk)
fixed_avg_itk = np.mean(fixed_volume_itk)
print('Image Parameters Fixed Volume (itk): ')
print(f'\tITK Origin: {fixed_volume_itk_metadata["origin"]}')
print(f'\tITK Spacing: {fixed_volume_itk_metadata["spacing"]}')
print(f'\tITK Direction: {fixed_volume_itk_metadata["direction"]}')
print(f'Average image intensity (ITK): {fixed_avg_itk}')
print(f'\tAverage image intensity (ITK): {fixed_avg_itk}')

## VFA FA=5
moving_volume_itk = itk.imread(moving_sequenceID, itk.F)
moving_volume_itk_metadata = dict(moving_volume_itk)
moving_avg_itk = np.mean(moving_volume_itk)
print('Image Parameters Moving Volume (itk): ')
print(f'\tITK Origin: {moving_volume_itk_metadata["origin"]}')
print(f'\tITK Spacing: {moving_volume_itk_metadata["spacing"]}')
print(f'\tITK Direction: {moving_volume_itk_metadata["direction"]}')
print(f'Average image intensity (ITK): {moving_avg_itk}')

## Dynamic FA=10
dynamic_volume_itk = itk.imread(dynamic_sequenceID, itk.F)
dynamic_avg_itk = np.mean(dynamic_volume_itk)
print(f'Dynamic (4d) volume average image intensity (ITK): {dynamic_avg_itk}')

## Extract the first volume from the dynamic sequence to build the moving 10deg:
# Retain only de first volume:
dynamic_nparray_itk = itk.GetArrayFromImage(dynamic_volume_itk)
if dynamic_sequence_pattern == 'Dyn':
    moving_10deg_nparray_itk = dynamic_nparray_itk[::nt, :, :]
elif dynamic_sequence_pattern == '4D':
    moving_10deg_nparray_itk = dynamic_nparray_itk[:ns, :, :]
    
# The metadata must be updated accordingly to create a valid ITK object:
moving_10deg_volume_itk = itk.GetImageFromArray(moving_10deg_nparray_itk)

# Get correct origin, spacing, direction
# Update spacing for 3D volume (slice spacing = spacing[0] * n_timepoints)
moving_10deg_volume_itk.SetSpacing(fixed_volume_itk.GetSpacing())
moving_10deg_volume_itk.SetOrigin(dynamic_volume_itk.GetOrigin())
moving_10deg_volume_itk.SetDirection(dynamic_volume_itk.GetDirection())
moving_10deg_volume_itk_metadata = dict(moving_10deg_volume_itk)
print('Image Parameters constructed volume from dynamic series')
print(f'\tITK Origin: {moving_10deg_volume_itk_metadata["origin"]}')
print(f'\tITK Spacing: {moving_10deg_volume_itk_metadata["spacing"]}')
print(f'\tITK Direction: {moving_10deg_volume_itk_metadata["direction"]}')

print(''.join(['§']*50))

print('Compare signal intensities ITK/DICOM')
print(f'Fixed volume ratio DICOM/ITK: {fixed_avg_dicom/fixed_avg_itk:.2f}')
print(f'Moving volume ratio DICOM/ITK: {moving_avg_dicom/moving_avg_itk:.2f}')
print(f'Dynamic (4d) volume ratio DICOM/ITK: {dynamic_avg_dicom/dynamic_avg_itk:.2f}')
print(''.join(['§']*50))

    
end_time = time.perf_counter()
elp_time = end_time - start_time
accrued_time += elp_time
print(f'§§§§ Loading data (ITK) took {elp_time:.2f}[s]')
print(''.join(['§']*50))


Load the data using ITK
Image Parameters Fixed Volume (itk): 
	ITK Origin: [ -80.86397552  -92.27231911 -179.24147975]
	ITK Spacing: [1.5        0.67961162 0.67961162]
	ITK Direction: [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
Average image intensity (ITK): 261.2562561035156
	Average image intensity (ITK): 261.2562561035156
Image Parameters Moving Volume (itk): 
	ITK Origin: [ -80.86397552  -92.27231911 -179.24147975]
	ITK Spacing: [1.5        0.67961162 0.67961162]
	ITK Direction: [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
Average image intensity (ITK): 245.40176391601562


ImageSeriesReader (0x36ce9ce40): Non uniform sampling or missing slices detected,  maximum nonuniformity:178.447



In [ ]:
print('********* Creating a working copy of the originals *********')
print('Please wait...')
start_time = time.perf_counter()

if loadMethod == 'itk':
    fixed_volume = itk.image_duplicator(fixed_volume_itk)
    moving_volume = itk.image_duplicator(moving_volume_itk)
    moving_10deg_volume = itk.image_duplicator(moving_10deg_volume_itk)

    # To check the dynamic series is still ok, saves the array directly using Nifti:
    # Create a NIfTI image object
    nifti_img = nib.Nifti1Image(np.permute_dims(dynamic_nparray_itk, [2, 1, 0]), eye4d)

elif loadMethod == 'dcm':
    fixed_volume = itk.image_duplicator(fixed_volume_dcm2itk)
    moving_volume = itk.image_duplicator(moving_volume_dcm2itk)
    moving_10deg_volume = itk.image_duplicator(moving_10deg_volume_dcm2itk)

    # To check the dynamic series is still ok, saves the array directly using Nifti:
    # Create a NIfTI image object
    nifti_img = nib.Nifti1Image(np.permute_dims(dynamic_volume_dcm, [3, 2, 1, 0]), eye4d)

# Save the NIfTI image to a file
nib.save(nifti_img, os.path.join(PRELIMPATH, 'qc_maps', f'{map_filename_pattern}_dynamic.nii.gz'))


itk.imwrite(fixed_volume, os.path.join(PRELIMPATH, 'qc_maps', f'{map_filename_pattern}_fixed_{fixed_sequence_pattern}.nii.gz'))
itk.imwrite(moving_volume, os.path.join(PRELIMPATH, 'qc_maps', f'{map_filename_pattern}_moving_{moving_sequence_pattern}.nii.gz'))
itk.imwrite(moving_10deg_volume, os.path.join(PRELIMPATH, 'qc_maps', f'{map_filename_pattern}_moving_{dynamic_sequence_pattern}.nii.gz'))

end_time = time.perf_counter()
elp_time = end_time - start_time
accrued_time += elp_time
print(f'§§§§ Saving data took {elp_time:.2f}[s]')


********* Creating a working copy of the originals *********
Please wait...
§§§§ Saving data took 33.86[s]
